In [1]:
import os
import requests
from datetime import datetime, date
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
from sqlalchemy import create_engine


# =========================================================

def get_urls(query, target_date):
    id = os.getenv("X-Naver-Client-Id")
    secret = os.getenv("X-Naver-Client-Secret")

    start = 1
    items = []
    while start < 1000:
        response = requests.get(
            url="https://openapi.naver.com/v1/search/news.json",
            headers={
                "X-Naver-Client-Id": id,
                "X-Naver-Client-Secret": secret
            },
            params={
                "query": query,
                "display": 100,
                "start": start,
                "sort": "date"
            }
        )
        data = response.json()
        temp_items = data['items']
        items.extend(temp_items)

        last_date = datetime.strptime(temp_items[-1]["pubDate"], '%a, %d %b %Y %H:%M:%S %z').replace(tzinfo=None).date()
        if last_date < target_date:
            break
        start += 100
    
    news_list = []
    for item in items:
        pub_date = datetime.strptime(item['pubDate'], '%a, %d %b %Y %H:%M:%S %z').replace(tzinfo=None).date()
        if pub_date < target_date:
            break
        link = item["link"]
        if not "n.news.naver" in link:
            continue
        news = {
            "title": item["title"],
            "link": link,
            "pub_date": pub_date
        }
        news_list.append(news)
    
    return news_list

# =========================================================

def setup_driver():
    """Chrome WebDriver 설정 및 초기화"""
    logs.append("try setup_driver")
    try:
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")  # 샌드박스 모드 비활성화 (리눅스 환경에서 필요할 수 있음)
        options.add_argument("--disable-dev-shm-usage")  # 공유 메모리 사용 제한 (리소스 문제 방지)
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

        driver = webdriver.Chrome(options=options) # Chrome 드라이버 실행
        logs.append("success setup_driver")
        return driver
    
    except Exception as e:
        logs.append(f"failed setup_driver {e}")


def crawl_news(driver, url):
    """웹사이트로 이동 및 로딩 대기"""
    logs.append("try navigate_to_page")
    try:
        driver.get(url) # 웹사이트 접속
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, "ct"))) # 페이지 요소 로딩 대기
        logs.append("success navigate_to_page")
        
    except Exception as e:
        logs.append(f"failed navigate_to_page {e}")

    logs.append("try scrape_news")
    try:
        content = driver.find_element(By.ID, "ct")
        dic_area =  content.find_element(By.ID, "dic_area")

        full_text = dic_area.get_attribute('innerText')
        try:
            media_end_summary = dic_area.find_element(By.CLASS_NAME, "media_end_summary").text
            img_desc = dic_area.find_element(By.CLASS_NAME, "img_desc").text
            text = full_text.replace(media_end_summary, "").strip()
            text = text.replace(img_desc, "").strip()
        except:
            text = full_text.strip()

        text = text.replace('\n', ' ')
        logs.append("success scrape_news")
        
        return text
    
    except Exception as e:
        logs.append(f"failed scrape_news: {e}")
        return ""

# =========================================================

def save_to_postgres(news_data):

    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST", "localhost")
    db_port = os.getenv("DB_PORT", "5432")
    db_name = os.getenv("DB_NAME")

    try:
        engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')
        df = pd.DataFrame(news_data)
        df.to_sql(name='news', con=engine, if_exists='append', index=False)
        
        logs.append(f"success save_to_postgres: {len(df)} rows")

    except Exception as e:
        logs.append(f"failed save_to_postgres: {e}")

# =========================================================

def save_logs(logs):
    log_filename = f"crawl_log_{datetime.date.today().strftime('%Y%m%d')}.txt"
    
    with open(log_filename, "a", encoding="utf-8") as f:
        f.write(f"\n--- Execution at {datetime.datetime.now()} ---\n")
        for log in logs:
            f.write(log + "\n")
    print(f"✅ 로그가 {log_filename}에 저장되었습니다.")

# =========================================================




In [6]:
# --- 네이버 뉴스 검색 API ---
target_date = date.today()
queries = ["국민연금", "공적연금", "주택연금", "퇴직연금"]

total_news = []
for query in queries:
    new_news = get_urls(query, target_date)
    total_news.extend(new_news)
total_news = list({news['link']:news for news in total_news}.values())

In [3]:
total_news

[{'title': '삼성바이오 노사갈등…전문가들 &quot;공급망·주주가치 리스크 확산&quot;',
  'link': 'https://n.news.naver.com/mnews/article/008/0005358339?sid=101',
  'pub_date': datetime.date(2026, 5, 15)},
 {'title': '[바이오사 리포트] 오스코텍, 로열티로 외형 확대…2Q 마일스톤 분기점',
  'link': 'https://n.news.naver.com/mnews/article/293/0000084713?sid=101',
  'pub_date': datetime.date(2026, 5, 15)},
 {'title': '노동·시민단체 &quot;국민<b>연금</b> 의결권 민간 위임 반대…대기업 거수기 전락...',
  'link': 'https://n.news.naver.com/mnews/article/014/0005522120?sid=102',
  'pub_date': datetime.date(2026, 5, 15)},
 {'title': '파업 강행한다는 삼전노조, 긴급조정권 발동 불가피하다 [논설실의 관점...',
  'link': 'https://n.news.naver.com/mnews/article/022/0004128421?sid=110',
  'pub_date': datetime.date(2026, 5, 15)},
 {'title': '반도체 호황이 만든 ‘역대급’ 초과세수…빚 vs 재분배 논쟁',
  'link': 'https://n.news.naver.com/mnews/article/119/0003090934?sid=101',
  'pub_date': datetime.date(2026, 5, 15)},
 {'title': '‘판교 기술’과 ‘전북 실증’ 만났다…경기-전북, 양자클러스터 ‘초...',
  'link': 'https://n.news.naver.com/mnews/article/666/0000107156?sid=102

In [7]:
# --- 뉴스 본문 크롤링 ---
logs = []

driver = setup_driver()

for news in total_news:
    url = news["link"]
    content = crawl_news(driver, url)
    news["text"] = content

driver.quit()

In [8]:
total_news

[{'title': '[바이오사 리포트] 오스코텍, 로열티로 외형 확대…2Q 마일스톤 분기점',
  'link': 'https://n.news.naver.com/mnews/article/293/0000084713?sid=101',
  'pub_date': datetime.date(2026, 5, 15),
  'text': '/구글 제미나이의 도움을 받아 시각화하고 기자가 최종 검토·확인해 제작한 이미지입니다. 오스코텍이 레이저티닙 로열티를 바탕으로 1분기 외형확대 흐름을 확인했다. 국내외 판매에 따른 로열티 수익이 매출을 끌어올렸지만 연구개발(R&D)비와 판매관리비가 함께 늘면서 영업손실은 되레 커졌다. 업계는 2분기 유럽 상업화 개시 마일스톤 반영 가능성을 단기 실적 변곡점으로 보고 있다. 레이저티닙에서 발생하는 수익이 후속 파이프라인의 조기 기술이전(LO) 확대와 항내성항암제 플랫폼 구축으로 이어질 수 있을지와 함께 지배구조 변수가 주목받는다.   로열티 유입에도 커진 손실 /구글 제미나이의 도움을 받아 시각화하고 기자가 최종 검토·확인해 제작한 그래픽입니다. 그래픽에 포함된 데이터와 내용은 기자가 직접 취재한 결과물입니다. 15일 금융감독원 전자공시시스템에 따르면 오스코텍은 1분기 연결기준 실적으로 매출 36억5000만원, 영업손실 100억원, 당기순손실 49억9000만원을 거뒀다. 매출이 전년동기 대비 88.4% 늘었지만 영업손실은 20% 확대됐다. 당기순손실은 35.7% 개선됐다. 2025년 1분기 실적은 매출 19억4000만원, 영업손실 83억2000만원, 당기순손실 77억7000만원이다.  시장에서는 로열티 유입이 외형을 키웠지만 R&D비를 흡수하기에는 아직 부족한 구조라는 분석이 나온다. 1분기 매출원가는 2025년 3억6000만원에서 2026년 5억원으로 느는 데 그쳤지만 같은 기간 판관비는 99억원에서 131억3000만원까지 확대됐다. 판관비 증가분 32억3000만원은 매출총이익 31억4000만원을 웃도는 수준이다. 이런 가운데 R&D비는 6

In [9]:




# --- DB 적재 및 로그 저장 ---
if total_news:
    save_to_postgres(total_news)
else:
    logs.append("No news data to save.")

save_logs(logs)

KeyboardInterrupt: 

In [ ]:
# pip install dotenv pandas selenium requests SQLAlchemy

In [43]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import requests
from datetime import datetime, date
from dotenv import load_dotenv
load_dotenv()

True

In [67]:
def get_urls(query, target_date):
    id = os.getenv("X-Naver-Client-Id")
    secret = os.getenv("X-Naver-Client-Secret")

    start = 1
    items = []
    while True:
        response = requests.get(
            url="https://openapi.naver.com/v1/search/news.json",
            # url = "https://openapi.naver.com/v1/search/news.xml" + encText # XML 결과
            headers={
                "X-Naver-Client-Id": id,
                "X-Naver-Client-Secret": secret
            },
            params={
                "query": query,
                "display": 100,
                "start": start,
                "sort": "date"
            }
        )
        data = response.json()
        temp_items = data['items']
        items.extend(temp_items)

        last_date = datetime.strptime(temp_items[-1]["pubDate"], '%a, %d %b %Y %H:%M:%S %z').replace(tzinfo=None).date()
        if last_date < target_date:
            break
        start += 100
    
    news_list = []
    for item in items:
        pub_date = datetime.strptime(item['pubDate'], '%a, %d %b %Y %H:%M:%S %z').replace(tzinfo=None).date()
        if pub_date < target_date:
            break
        link = item["link"]
        if not "n.news.naver" in link:
            continue
        news = {
            "title": item["title"],
            "link": link,
            "pub_date": pub_date
        }
        news_list.append(news)
    
    return news_list

In [68]:
target_date = date.today()
query = "주택연금"
total_news = get_urls(query, target_date)

In [5]:
total_news

[{'title': '삼성바이오 노사갈등…전문가들 &quot;공급망·주주가치 리스크 확산&quot;',
  'link': 'https://n.news.naver.com/mnews/article/008/0005358339?sid=101',
  'pub_date': datetime.date(2026, 5, 15),
  'text': '성과급을 둘러싼 삼성그룹 주요 계열사 노사 갈등이 단순 임금단체협상(임단협) 문제를 넘어 공급망과 기업 및 주주가치 전반에 영향을 미칠 수 있는 이슈로 확산하고 있다는 진단이 나왔다. 전문가들은 노동자의 정당한 보상 요구는 자연스러운 흐름이라고 평가했지만, 영업이익 연동 성과급의 명문화와 생산 중단을 전제로 한 투쟁은 기업 경쟁력과 이해관계자 전반에 부담으로 작용할 수 있다고 지적했다. 사측 역시 구성원과의 소통 및 보상 체계 관리에 보다 적극적으로 나서야 한다는 제언도 나왔다.  주주행동연구원은 15일 오전 서울 중구 한국프레스센터에서 \'주주 관점에서 본 최근의 파업 이슈: 삼성그룹 사례를 중심으로\'를 주제로 전문가 좌담회를 개최했다. 주주행동연구원 원장인 강원 세종대 경영학부 교수가 좌장을 맡았으며 △강승훈 인하대 바이오제약공학과 교수 △권재열 경희대 법학전문대학원 교수 △송헌재 서울시립대 경제학부 교수 △이승길 한국ILO협회 회장 △정무권 국민대 경영대 교수 등이 패널로 참석했다.  강원 교수는 모두발언에서 최근 노사 갈등 양상이 기존 임금 인상 중심에서 영업이익 배분 구조를 둘러싼 갈등으로 변화하고 있다고 진단했다. 그는 "과거에는 임금을 얼마나 인상할 것인가가 중심이었다면 최근에는 회사가 큰 이익을 냈을 때 그 과실을 어떤 기준으로 누구와 나눌 것인가가 새로운 쟁점으로 떠오르고 있다"고 말했다.  이어 삼성바이오로직스 노조가 영업이익 20% 성과급 배분과 1인당 3000만원 격려금 등을 요구하고 있는 점을 언급하며 "노사 문제가 이제는 소액주주와 협력업체, 국민연금 가입자 등 다양한 이해관계자와 충돌하는 구조로 바뀌고 있

### 본문 크롤링

In [25]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [76]:
logs = []

In [28]:
def setup_driver():
    """Chrome WebDriver 설정 및 초기화"""
    logs.append("try setup_driver")
    try:
        options = webdriver.ChromeOptions()
        options.add_argument("--no-sandbox")  # 샌드박스 모드 비활성화 (리눅스 환경에서 필요할 수 있음)
        options.add_argument("--disable-dev-shm-usage")  # 공유 메모리 사용 제한 (리소스 문제 방지)
        
        driver = webdriver.Chrome(options=options) # Chrome 드라이버 실행
        logs.append("success setup_driver")
        return driver
    
    except Exception as e:
        logs.append(f"failed setup_driver {e}")

In [77]:
def crawl_news(driver, url):
    """웹사이트로 이동 및 로딩 대기"""
    logs.append("try navigate_to_page")
    try:
        driver.get(url) # 웹사이트 접속

        # 페이지 요소 로딩 대기
        WebDriverWait(driver, 1).until(EC.presence_of_element_located((By.ID, "ct")))  
        logs.append("success navigate_to_page")
        
    except Exception as e:
        logs.append(f"failed navigate_to_page {e}")

    logs.append("try scrape_news")
    try:
        content = driver.find_element(By.ID, "ct")
        dic_area =  content.find_element(By.ID, "dic_area")

        full_text = dic_area.get_attribute('innerText')
        try:
            media_end_summary = dic_area.find_element(By.CLASS_NAME, "media_end_summary").text
            img_desc = dic_area.find_element(By.CLASS_NAME, "img_desc").text
            text = full_text.replace(media_end_summary, "").strip()
            text = full_text.replace(img_desc, "").strip()
        except:
            text = full_text.strip()

        text = text.replace('\n', ' ')
        logs.append("success scrape_news")
        
        return text
    
    except Exception as e:
        logs.append(f"failed scrape_news: {e}")
        return ""

In [ ]:
driver = setup_driver()
contents = []

for news in total_news:
    url = news["link"]
    content = crawl_news(driver, url)
    news["text"] = content

driver.quit() # 드라이버 종료

In [79]:
for idx, text in enumerate(contents):
    print(idx, text[:100])

0 향후 5년 투자방향 결정 앞둔 국민연금 민노총 “연금, 수익률만 좇아선 지속가능성 못 지켜” 공공임대·돌봄·의료 등 사회인프라 투자 요구 “주주권 행사도 가입자 이익 중심으로 강화
1 청년주택 3000호 공급 등 청년공약 도지사 직속 청년담당관·청년 센터    더불어민주당 김경수 경남지사 후보가 “일자리·창업·주거 등 분야에서 경남이 청년의 든든한 언덕 되겠다”
2 <이광재 더불어민주당 하남갑 국회의원 후보> - 하남, 강남·판교와 붙어 있어 미래산업 최적지 - 국공유지 활용해 ‘초순수’ 등 첨단산업 유치해야 - AI시대엔 지친 영혼 쉴 녹색
3 ■AI 프리즘 [금융상품 뉴스] 美국채 30년물 금리 19년 만에 5% 돌파 코스피 활황 10대 증권사 1Q 순익 114%↑ 반도체 ETF 나란히 순자산 2조 돌파    ▲ AI 
4 ■AI프리즘 [부동산 뉴스] 성북 0.54%·서대문 0.45% 뛰어 비아파트 회복, 상업용 빌딩 부진 강남 개발·재건축 경쟁 본격화    ▲ AI 프리즘* 맞춤형 경제 브리핑 * 
5 청장년 가입률 고령층 대비 7분의 1 수준…1인 가구 사각지대 해소 주력 카카오톡 채널 신설로 안내부터 신청까지 한 번에…안내 사업 163종으로 확대    한국사회보장정보원 본원 
6 경기도 구리시에 사는 박모(71)씨는 최근 주택연금을 두고 고심하고 있다. 시세 7억3000만원 상당의 아파트로 연금을 받으면 월평균 약 230만원(정액형 기준) 받을 수 있어 가
7 ‘요람에서 무덤까지’로 표현되는 복지국가의 대명사이던 스웨덴이 민영화와 감세로 경제 활력을 되찾았다는 소식이다. 스웨덴의 1차 병원 절반이 민간 소유이고, 그 대다수를 사모펀드회사
